# 🧠 Taller 5 SQL - Lógica en la BD PostgreSQL
## Gimnasia Rítmica

Este taller está diseñado para ejecutarse en Google Colaboratory usando una base de datos PostgreSQL. Aprenderás immplementar lógica en la BD, usandovistas, triggers y procedimientos almacenados

## Objetivos
- Crear una base de datos PostgreSQL en Colab.
- Crear vistas
- Crear tablas temporales
- Crear triggers al insertarse datos en la BD
- Implementar Procedimientos almacenados, que usen vistas, triggers y actualizand datos


In [1]:
# Instalación y configuración de PostgreSQL
!apt update
!apt install postgresql postgresql-contrib -y
!service postgresql start
!sudo -u postgres psql -c "CREATE USER root WITH SUPERUSER"


Get:1 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Hit:2 https://cli.github.com/packages stable InRelease
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:7 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,064 kB]
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,325 kB]
Hit:11 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:12 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Hit:13 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu 

In [2]:
# Cargar extensión SQL y conectar con PostgreSQL
%load_ext sql
%config SqlMagic.feedback=False
%config SqlMagic.autopandas=True
%sql postgresql+psycopg2://@/postgres


## 🏗️ Crear tablas

Creamos las tablas necesarias para modelar la base de datos de gimnasia rítmica. Incluye campos tipo `DATE` y relaciones entre entidades.


In [3]:
%%sql

DROP TABLE IF EXISTS Evaluaciones, Presentaciones, Gimnastas, Conjuntos, Instrumentos, Campeonatos, Tipos, Equipos CASCADE;

CREATE TABLE Equipos (
    id_equipo SERIAL PRIMARY KEY,
    nombre_equipo TEXT
);

CREATE TABLE Conjuntos (
    id_conj SERIAL PRIMARY KEY,
    nombre_conj TEXT,
    id_equipo INTEGER REFERENCES Equipos(id_equipo)
);

CREATE TABLE Gimnastas (
    id_gim SERIAL PRIMARY KEY,
    nombre TEXT,
    fecha_nac DATE,
    id_conj INTEGER REFERENCES Conjuntos(id_conj),
    id_equipo INTEGER REFERENCES Equipos(id_equipo)
);

CREATE TABLE Instrumentos (
    id_inst SERIAL PRIMARY KEY,
    nombre_inst TEXT
);

CREATE TABLE Campeonatos (
    id_camp SERIAL PRIMARY KEY,
    nombre_camp TEXT,
    fecha_camp DATE
);

CREATE TABLE Tipos (
    id_tipo SERIAL PRIMARY KEY,
    nombre_tipo TEXT
);

CREATE TABLE Presentaciones (
    id_pres SERIAL PRIMARY KEY,
    id_gim INTEGER REFERENCES Gimnastas(id_gim),
    id_conj INTEGER REFERENCES Conjuntos(id_conj),
    id_inst INTEGER REFERENCES Instrumentos(id_inst),
    id_camp INTEGER REFERENCES Campeonatos(id_camp),
    id_tipo INTEGER REFERENCES Tipos(id_tipo)
);

CREATE TABLE Evaluaciones (
    id_eval SERIAL PRIMARY KEY,
    id_pres INTEGER REFERENCES Presentaciones(id_pres),
    pje_eje REAL,
    pje_dif REAL,
    pje_art REAL
);


 * postgresql+psycopg2://@/postgres


""


In [4]:
%config SqlMagic.style = '_DEPRECATED_DEFAULT'


## 📥 Insertar datos simulados

Insertamos datos ficticios para poblar la base de datos: 5 equipos, 10 conjuntos, 20 gimnastas, 15 campeonatos, 50 presentaciones individuales, 50 presentaciones de conjunto y 100 evaluaciones.


In [5]:
%%sql

-- Insertar equipos
INSERT INTO Equipos (nombre_equipo) VALUES
('Estrellas'), ('Auroras'), ('Fénix'), ('Orion'), ('Galaxia');

-- Insertar conjuntos
INSERT INTO Conjuntos (nombre_conj, id_equipo) VALUES
('Estrellas A', 1), ('Estrellas B', 1), ('Auroras A', 2), ('Auroras B', 2),
('Fénix A', 3), ('Fénix B', 3), ('Orion A', 4), ('Orion B', 4),
('Galaxia A', 5), ('Galaxia B', 5);

-- Insertar gimnastas
INSERT INTO Gimnastas (nombre, fecha_nac, id_conj, id_equipo) VALUES
('Ana Torres', '2005-03-12', 1, 1), ('Lucía Pérez', '2006-07-25', 1, 1),
('María Gómez', '2004-11-05', 2, 1), ('Sofía Díaz', '2007-01-20', 2, 1),
('Valentina Ruiz', '2005-06-30', 3, 2), ('Camila Soto', '2006-09-15', 3, 2),
('Isabella León', '2004-12-01', 4, 2), ('Martina Ríos', '2007-04-10', 4, 2),
('Renata Silva', '2005-08-22', 5, 3), ('Emilia Vargas', '2006-10-05', 5, 3),
('Josefa Herrera', '2004-07-18', 6, 3), ('Antonia Castro', '2007-02-28', 6, 3),
('Florencia Peña', '2005-05-14', 7, 4), ('Amanda Fuentes', '2006-11-11', 7, 4),
('Julieta Navarro', '2004-09-09', 8, 4), ('Agustina Bravo', '2007-03-03', 8, 4),
('Daniela Pino', '2005-12-25', 9, 5), ('Bianca Morales', '2006-06-06', 9, 5),
('Carla Espinoza', '2004-10-10', 10, 5), ('Fernanda Reyes', '2007-07-07', 10, 5),
('Colomba Gómez', '2004-11-05', 2, 1), ('Montserrat Díaz', '2007-01-20', 2, 1);

-- Insertar instrumentos
INSERT INTO Instrumentos (nombre_inst) VALUES
('Cinta'), ('Aro'), ('Balón'), ('Manos libres'), ('Cuerda');

-- Insertar campeonatos
INSERT INTO Campeonatos (nombre_camp, fecha_camp) VALUES
('Campeonato Nacional', '2023-06-15'), ('Copa Primavera', '2023-09-10'),
('Copa Invierno', '2023-12-05'), ('Torneo Sur', '2023-08-20'),
('Copa Norte', '2023-07-01'), ('Copa Andes', '2023-10-10'),
('Copa Pacífico', '2023-11-11'), ('Copa Estrellas', '2024-05-05'),
('Copa Aurora', '2024-04-04'), ('Copa Fénix', '2025-03-03'),
('Copa Orion', '2025-02-02'), ('Copa Galaxia', '2025-01-01'),
('Copa Final', '2024-12-31'), ('Copa Apertura', '2025-01-15'),
('Copa Clausura', '2024-12-01');

-- Insertar tipos
INSERT INTO Tipos (nombre_tipo) VALUES ('Individual'), ('Conjunto');




 * postgresql+psycopg2://@/postgres


""


In [6]:
%%sql
-- Insertar presentaciones
DO $$
DECLARE
    i INTEGER := 1;
    gim_id INTEGER;
    conj_id INTEGER;
    inst_id INTEGER;
    tipo_id INTEGER;
    camp_id INTEGER;
BEGIN
    WHILE i <= 100 LOOP
        -- Selección aleatoria de IDs válidos
        SELECT id_gim INTO gim_id FROM Gimnastas ORDER BY RANDOM() LIMIT 1;
        SELECT id_conj INTO conj_id FROM Conjuntos ORDER BY RANDOM() LIMIT 1;
        SELECT id_inst INTO inst_id FROM Instrumentos ORDER BY RANDOM() LIMIT 1;
        SELECT id_tipo INTO tipo_id FROM Tipos ORDER BY RANDOM() LIMIT 1;
        SELECT id_camp INTO camp_id FROM Campeonatos ORDER BY RANDOM() LIMIT 1;

        -- Inserción en la tabla Presentaciones
        INSERT INTO Presentaciones (id_gim, id_conj, id_inst, id_camp, id_tipo)
        VALUES (gim_id, conj_id, inst_id, camp_id, tipo_id);

        i := i + 1;
    END LOOP;
END $$;


 * postgresql+psycopg2://@/postgres


""


In [7]:
%%sql
-- Insertar evaluaciones
INSERT INTO Evaluaciones (id_pres, pje_eje, pje_dif, pje_art)
SELECT p.id_pres,
       ROUND((RANDOM() * 5 + 5)::NUMERIC, 2),
       ROUND((RANDOM() * 5 + 5)::NUMERIC, 2),
       ROUND((RANDOM() * 5 + 5)::NUMERIC, 2)
FROM Presentaciones p
LIMIT 100;

 * postgresql+psycopg2://@/postgres


""


## 🔍 Ejercicios de Consultas SQL



1.- Cree una vista que entregue los lugares obtenidos en cada campeonato para la competencia individual, separado por tipo de presentación e instrumento, siendo el primer lugar el que haya obtenido mayor puntaje total.

In [14]:
%%sql
CREATE VIEW lugares_individual_2 AS
SELECT
    c.nombre_camp,
    t.nombre_tipo,
    i.nombre_inst,
    g.nombre AS nombre_gimnasta,
    e.pje_eje + e.pje_dif + e.pje_art AS puntaje_total,
    RANK() OVER (PARTITION BY c.nombre_camp, t.nombre_tipo, i.nombre_inst ORDER BY (e.pje_eje + e.pje_dif + e.pje_art) DESC) as lugar
FROM Evaluaciones e
JOIN Presentaciones p ON e.id_pres = p.id_pres
JOIN Campeonatos c ON p.id_camp = c.id_camp
JOIN Tipos t ON p.id_tipo = t.id_tipo
JOIN Instrumentos i ON p.id_inst = i.id_inst
JOIN Gimnastas g ON p.id_gim = g.id_gim
WHERE t.nombre_tipo = 'Individual';

 * postgresql+psycopg2://@/postgres


""


In [19]:
%%sql
SELECT *
FROM lugares_individual_2;

 * postgresql+psycopg2://@/postgres


,nombre_camp,nombre_tipo,nombre_inst,nombre_gimnasta,puntaje_total,lugar
0,Campeonato Nacional,Individual,Cinta,María Gómez,21.420000,1
1,Campeonato Nacional,Individual,Cinta,Renata Silva,20.290000,2
2,Campeonato Nacional,Individual,Cuerda,Josefa Herrera,20.890000,1
3,Campeonato Nacional,Individual,Manos libres,María Gómez,24.810000,1
4,Copa Andes,Individual,Cinta,Sofía Díaz,22.480000,1
5,Copa Andes,Individual,Manos libres,Carla Espinoza,24.050001,1
6,Copa Andes,Individual,Manos libres,Carla Espinoza,23.740000,2
7,Copa Andes,Individual,Manos libres,Colomba Gómez,23.430000,3
8,Copa Aurora,Individual,Aro,Montserrat Díaz,22.200000,1
9,Copa Aurora,Individual,Cinta,Amanda Fuentes,23.080000,1


2.- Use la vista creada en el paso anterior y obtenga los lugares para el campenoato **Copa Pacífico**

In [16]:
%%sql
SELECT *
FROM lugares_individual
WHERE nombre_camp = 'Copa Pacífico';

 * postgresql+psycopg2://@/postgres


,nombre_camp,nombre_tipo,nombre_inst,nombre_gimnasta,puntaje_total,lugar
0,Copa Pacífico,Individual,Aro,Renata Silva,24.22,1
1,Copa Pacífico,Individual,Aro,Antonia Castro,20.99,2
2,Copa Pacífico,Individual,Cinta,Julieta Navarro,26.14,1
3,Copa Pacífico,Individual,Cinta,Carla Espinoza,23.75,2
4,Copa Pacífico,Individual,Cinta,Colomba Gómez,21.40,3


3.- Cree un procedimiento almacenado que reciba como parámetro el nombre un equipo y lo inserte en la tabla equipos

In [18]:
%%sql

CREATE OR REPLACE PROCEDURE insertar_equipo(p_nombre_equipo TEXT)
LANGUAGE plpgsql
AS $$
BEGIN
    INSERT INTO Equipos (nombre_equipo)
    VALUES (p_nombre_equipo);
END;
$$;

 * postgresql+psycopg2://@/postgres


""


4.- Usando el procedimiento creado en el paso anterior, cree 2 equipos nuevos y verifique que se ejecutó correctamente

In [ ]:
%%sql


In [ ]:
%%sql


5.- Cree un procedimiento almacenado que reciba como parámetros el nombre de un equipo, el nombre de una gimnasta y su fecha de nacimiento. El procedimiento busca el id del equipo y crea la gimnasta en la tabla de gimnastas. En caso de no existir el equipo, entrega un mensaje de error indicando que el equipo no fue encontrado

In [ ]:
%%sql

$$

6.- Use el procedimiento almacenado creado en el paso anterior y ejecútelo para los siguientes 2 casos y compruebe que funcione correctamente


1.   Use uno de los equipos creados en el paso 4
2.   Use un equipo que no exista



In [ ]:
%%sql


In [ ]:
%%sql


7.- Realice los siguientes pasos:
*   Cree una tabla que se llama tmp_lugares_invidual, que tenga los mismos campos y tipos de datos que devuelve la vista *lugares_individual*, más un campo tipo timestamp
*   Cree un trigger que cada_vez que se inserta un registro en la tabla de ***Evaluaciones***, inserte el resultado de la ejecución de la vista creada en el paso 1, en la tabla tmp_lugares_individual y en el campo timestamp la fecha y hora en que se ejecutó el trigger.

In [ ]:
%%sql


In [ ]:
%%sql


In [ ]:
%%sql


In [ ]:
%%sql



8.- Cree 2 registros en la tabla **Evaluaciones** para distintos campeonatos y verifique que funcionó correctamente

In [ ]:
%%sql


In [ ]:
%%sql

